In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Folder containing full-length propensity files
folder_path = "Code_simulation_analysis/analysis/helical_propensity/Full_length/"

files = [
    "propensities_1.00.csv",
    "propensities_0.98.csv",
    "propensities_0.95.csv",
    "propensities_0.93.csv",
    "propensities_0.91.csv",
    "propensities_0.89.csv",
    "propensities_0.86.csv",
    "propensities_0.84.csv",
    "propensities_0.82.csv",
    "propensities_0.80.csv",
    "propensities_0.77.csv",
    "propensities_0.75.csv"
]

# Extract H-bond strength from filenames
hbs_values = [float(f.split('_')[1].replace('.csv', '')) * 100 for f in files]

# Define segments
segments = {
    "H0": (3, 22),
    "H1": (26, 80),
    "H2": (84, 136),
    "H3": (137, 156),
    "H4": (164, 188),
    "H5": (192, 215),
    "H6": (251, 264)
}

# Store mean and SEM
segment_means = {key: [] for key in segments.keys()}
segment_sems  = {key: [] for key in segments.keys()}

# ----------------------------
# Extract segment statistics
# ----------------------------
for file in files:
    file_path = os.path.join(folder_path, file)
    
    if not os.path.exists(file_path):
        print(f"⚠ File not found: {file}")
        continue
    
    data = pd.read_csv(file_path)
    
    for seg_name, (start, end) in segments.items():
        seg_data = data[(data["Residue"] >= start) & (data["Residue"] <= end)]
        
        values = seg_data["Average Propensity"].values
        stds   = seg_data["Standard Deviation"].values
        
        N = len(values)
        
        mean_value = np.mean(values)
        
        # Proper error propagation
        sem_value = np.sqrt(np.sum(stds**2)) / N
        
        segment_means[seg_name].append(mean_value)
        segment_sems[seg_name].append(sem_value)

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.tab10(np.linspace(0, 1, len(segments)))

for (seg_name, means), color in zip(segment_means.items(), colors):
    ax.errorbar(
        hbs_values,
        means,
        yerr=segment_sems[seg_name],
        fmt='o-',
        linewidth=2.5,
        markersize=7,
        capsize=4,
        label=seg_name,
        color=color
    )

# Formatting
ax.set_xlabel("H-bond Strength (%)", fontsize=30, fontweight='bold')
ax.set_ylabel("Helical Propensity", fontsize=30, fontweight='bold')

ax.tick_params(axis='both', labelsize=26)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight('bold')

ax.set_ylim(0, 1.0)

legend = ax.legend( fontsize=26, title_fontsize=18, frameon=False)
for text in legend.get_texts():
    text.set_fontweight('bold')
legend.get_title().set_fontweight('bold')

# Full box with thicker border
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.rcParams['axes.linewidth'] = 2

plt.tight_layout()

#output_path = os.path.join( "IM30 segment_average_vs_HBS_with_error.png")
#plt.savefig(output_path, dpi=600)
print(f"Plot saved to: {output_path}")

plt.show()
